# Models and evaluation

The same training, validation, and test split is reused. The test set was held back until the final evaluation.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

root = Path.cwd()
if not (root / 'datasets').exists():
    root = root.parent
raw = root / 'datasets' / 'hdfs_v1' / 'preprocessed'
processed = root / 'datasets' / 'processed'
figures = root / 'reports' / 'figures'

comparison = pd.read_csv(root / 'reports' / 'tables' / 'model_comparison.csv')
final = pd.read_csv(root / 'reports' / 'tables' / 'final_test_results.csv')
print('Validation comparison')
display(comparison)
print('Final test results')
display(final)


Validation comparison


,model,threshold,accuracy,precision,recall,f1,auc,log_loss,tn,fp,fn,tp,validation_cost
0,Random forest,0.407487,0.999113,0.970868,0.999703,0.985075,0.999910,0.092840,111544,101,1,3366,111
1,MLP,0.194174,0.999157,0.973638,0.998218,0.985775,0.999991,0.001058,111554,91,6,3361,151
2,Logistic regression balanced,0.883553,0.998774,0.961121,0.998515,0.979461,0.999662,0.006945,111509,136,5,3362,186
3,Logistic regression,0.107808,0.998809,0.962486,0.998218,0.980026,0.998916,0.002643,111514,131,6,3361,191
4,Decision tree,0.727783,0.998626,0.958036,0.996733,0.977001,0.999035,0.006451,111498,147,11,3356,257
5,Naive Bayes,1.000000,0.970725,0.000000,0.000000,0.000000,0.997561,0.049972,111645,0,3367,0,452
6,kNN,1.000000,0.970725,0.000000,0.000000,0.000000,0.998911,0.003018,111645,0,3367,0,1140
7,SVM,0.925412,0.997826,0.993352,0.931987,0.961692,0.997363,0.005577,111624,21,229,3138,2311
8,AdaBoost,0.470737,0.993862,0.882437,0.911791,0.896874,0.998077,0.409426,111236,409,297,3070,3379
9,PCA anomaly detector,0.986770,0.977272,0.593448,0.710128,0.646566,0.957503,1.352392,110007,1638,976,2391,11398


Final test results


,model,threshold,accuracy,precision,recall,f1,auc,log_loss,tn,fp,fn,tp
0,Random forest,0.407487,0.999426,0.981341,0.999406,0.990291,0.999945,0.092249,111581,64,2,3366
1,MLP,0.194174,0.999261,0.976764,0.998515,0.987520,0.999734,0.000722,111565,80,5,3363
2,Logistic regression balanced,0.883553,0.998939,0.965310,0.999703,0.982205,0.999691,0.005894,111524,121,1,3367


All models used the same validation cost rule. A missed anomaly costs 10 times a false alarm, so thresholds were selected by the lowest validation cost.

In [2]:
cost_columns = ['model', 'threshold', 'validation_cost', 'precision', 'recall', 'f1']
display(comparison[cost_columns])
print('Best validation model:', comparison.iloc[0]['model'])
print('Best validation threshold:', comparison.iloc[0]['threshold'])


,model,threshold,validation_cost,precision,recall,f1
0,Random forest,0.407487,111,0.970868,0.999703,0.985075
1,MLP,0.194174,151,0.973638,0.998218,0.985775
2,Logistic regression balanced,0.883553,186,0.961121,0.998515,0.979461
3,Logistic regression,0.107808,191,0.962486,0.998218,0.980026
4,Decision tree,0.727783,257,0.958036,0.996733,0.977001
5,Naive Bayes,1.000000,452,0.000000,0.000000,0.000000
6,kNN,1.000000,1140,0.000000,0.000000,0.000000
7,SVM,0.925412,2311,0.993352,0.931987,0.961692
8,AdaBoost,0.470737,3379,0.882437,0.911791,0.896874
9,PCA anomaly detector,0.986770,11398,0.593448,0.710128,0.646566


Best validation model: Random forest
Best validation threshold: 0.4074870317389577


The comparison includes PCA reconstruction error, logistic regression, decision tree, kNN, Naive Bayes, linear SVM, random forest, AdaBoost, and MLP. kNN and SVM used the documented fixed stratified training samples.

In [3]:
models = comparison['model'].tolist()
print(models)
print('kNN training sample: 15000')
print('SVM training sample: 10000')
print('All other models used the full training partition.')


['Random forest', 'MLP', 'Logistic regression balanced', 'Logistic regression', 'Decision tree', 'Naive Bayes', 'kNN', 'SVM', 'AdaBoost', 'PCA anomaly detector']
kNN training sample: 15000
SVM training sample: 10000
All other models used the full training partition.


Accuracy is not enough because the anomaly class is small. Precision, recall, F1, AUC, log loss, and the confusion-matrix counts are shown together.

In [4]:
metrics = ['model', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'log_loss', 'tn', 'fp', 'fn', 'tp']
display(final[metrics])


,model,accuracy,precision,recall,f1,auc,log_loss,tn,fp,fn,tp
0,Random forest,0.999426,0.981341,0.999406,0.990291,0.999945,0.092249,111581,64,2,3366
1,MLP,0.999261,0.976764,0.998515,0.987520,0.999734,0.000722,111565,80,5,3363
2,Logistic regression balanced,0.998939,0.965310,0.999703,0.982205,0.999691,0.005894,111524,121,1,3367


The final comparison was selected on validation data before the test set was read. Random forest has the highest final F1 in the saved final test table.

In [5]:
best = final.sort_values('f1', ascending=False).iloc[0]
print(best['model'])
print(best['f1'])
print(best['precision'])
print(best['recall'])


Random forest
0.9902912621359224
0.98134110787172
0.9994061757719715


Conclusion and limitations

The strongest saved result is useful on this labelled HDFS benchmark, but it does not prove that the same threshold or model will work on another Hadoop system. The feature values are log-event counts, so the models explain association with known labels rather than the cause of a failure.